# 31. 분포 진단 — 결과가 "좋은 모양"인가

슬라이드 「분포조절」 · 「필터적합성 조사」 · 「코퍼스 분포 조사」

정확성은 판정하기 어렵다. 대신 **결과 유사도의 모양**을 보면 관리할 수 있는 신호가 나온다.
1. 좋은 분포: 매칭된 항목과 아닌 항목의 점수가 확실히 갈린다 (무릎점이 뚜렷)
2. 필터 적합성: 필터로 좁힌 집합이 질의와 의미가 맞는가
3. 코퍼스 분포: 특정 문서·출처로 결과가 쏠리지 않는가

30번의 검색 결과(`results/klue_runs.pkl`)를 재사용한다.

In [ ]:
from labutil import *
import pickle

conn = connect()
assert os.path.exists("results/klue_runs.pkl"), "30번을 먼저 실행한다"
runs = pickle.load(open("results/klue_runs.pkl", "rb"))
QIDS = sorted(runs["gold"])
sims = {qid: np.array([s for _, s in runs["vec_chunk"][qid]]) for qid in QIDS}
rank = {qid: ([d for d, _ in runs["vec_chunk"][qid]].index(runs["gold"][qid]) + 1
              if runs["gold"][qid] in dict(runs["vec_chunk"][qid]) else None) for qid in QIDS}

## 1. 유사도 곡선 — 맞힌 질문과 틀린 질문

In [ ]:
ok = [qid for qid in QIDS if rank[qid] == 1][:4]
bad = [qid for qid in QIDS if rank[qid] is None][:4]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), sharey=True)
for ax, group, title in [(axes[0], ok, "정답이 1위인 질문"), (axes[1], bad, "정답이 50위 안에 없는 질문")]:
    for qid, c in zip(group, ["#FF7043", "#009CA6", "#3D484A", "#84A59D"]):
        ax.plot(range(1, 51), sims[qid], "-", color=c, lw=1.8)
        if rank[qid]:
            ax.scatter([rank[qid]], [sims[qid][rank[qid] - 1]], s=120, marker="*", color=c, zorder=3)
    ax.set_xlabel("순위"); ax.set_title(title)
axes[0].set_ylabel("코사인 유사도 (청크 기준)")
plt.tight_layout(); plt.show()

**관찰:** 맞힌 질문은 1위가 뚝 떨어져 나와 있다. 틀린 질문은 곡선이 **밋밋하게** 내려온다 — 어디서 자를지 알 수 없는 "나쁜 분포".

## 2. 무릎점 찾기 — 2차 미분 vs 최대 간격 vs Kneedle
슬라이드는 "유사도 값을 2차 미분해 변곡점을 측정"을 제안했다. 이산 곡선에서 세 방법을 비교한다.

In [ ]:
def cut_gap(s):          # 인접한 두 점수 차이(1차 차분)가 가장 큰 곳
    return int(np.argmax(-np.diff(s))) + 1


def cut_d2(s):           # 2차 차분이 가장 큰 곳
    return int(np.argmax(np.diff(s, 2))) + 1


def cut_kneedle(s):      # 양 끝을 잇는 직선에서 가장 멀리 떨어진 곳 (Kneedle)
    x = np.linspace(0, 1, len(s)); y = (s - s.min()) / (s.max() - s.min() + 1e-12)
    return int(np.argmax((1 - x) - y)) + 1


methods = {"최대 간격 (1차 차분)": cut_gap, "2차 차분": cut_d2, "Kneedle": cut_kneedle}
cuts = pd.DataFrame({name: [f(sims[qid]) for qid in QIDS] for name, f in methods.items()}, index=QIDS)

qex = ok[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].plot(range(1, 51), sims[qex], "-o", ms=3, color="#9aa5a8")
for (name, f), c in zip(methods.items(), ["#FF7043", "#3D484A", "#009CA6"]):
    axes[0].axvline(f(sims[qex]) + 0.5, color=c, lw=2, label=f"{name}: 상위 {f(sims[qex])}개")
axes[0].legend(); axes[0].set_xlabel("순위"); axes[0].set_title("한 질문에서 세 방법이 자르는 위치")
bins = np.arange(0.5, 51, 1)
for (name, _), c in zip(methods.items(), ["#FF7043", "#3D484A", "#009CA6"]):
    axes[1].hist(cuts[name], bins=bins, alpha=0.5, color=c, label=name)
axes[1].set_xlabel("자른 위치 (상위 몇 개를 남기나)"); axes[1].set_ylabel("질문 수"); axes[1].legend()
axes[1].set_title("질문 500개에서 자른 위치의 분포")
plt.tight_layout(); plt.show()

kept = pd.DataFrame({name: {"평균 남긴 개수": cuts[name].mean(),
                            "정답이 남은 비율": np.mean([rank[qid] is not None and rank[qid] <= cuts.loc[qid, name] for qid in QIDS])}
                     for name in methods}).T
kept.round(3)

**관찰:**
- 최대 간격과 2차 차분은 대부분 **1~2위 바로 뒤**에서 자른다. 1위 하나가 튀어나온 곡선이 많기 때문
- 2차 차분은 가끔 곡선 중간의 작은 요철에 반응해 엉뚱한 곳(10~20위)에서 자른다 — 이산 점수에서는 잡음에 민감하다
- Kneedle 은 곡선 전체 모양을 보고 평균 12개를 남긴다. 정답이 잘린 쪽에 남는 비율이 0.52 로 가장 높다
- 어느 방법이 옳다기보다 **목적에 맞게** 고른다: "확실한 것만"(간격) vs "후보를 넉넉히"(Kneedle)

## 3. 분포가 좋으면 믿을 만한가 — 분리도로 정답 예측
분리도 = 1위 점수 − 2~10위 평균. 분리도가 큰 질문부터 답한다면 정확도는?

In [ ]:
sep = pd.Series({qid: sims[qid][0] - sims[qid][1:10].mean() for qid in QIDS})
top1 = pd.Series({qid: rank[qid] == 1 for qid in QIDS})
b = {qid: np.array([s for _, s in runs["bm25"][qid]]) for qid in QIDS}
sep_b = pd.Series({qid: (b[qid][0] - b[qid][1:10].mean()) / b[qid][0] for qid in QIDS})
top1_b = pd.Series({qid: runs["bm25"][qid][0][0] == runs["gold"][qid] for qid in QIDS})

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
axes[0].boxplot([sep[top1], sep[~top1]], tick_labels=["정답 1위", "정답 1위 아님"], widths=0.5)
axes[0].set_ylabel("분리도 (1위 - 2~10위 평균)"); axes[0].set_title("벡터 검색: 맞힌 질문의 분리도가 크다")
for s_, t_, name, c in [(sep, top1, "벡터 (청크)", "#FF7043"), (sep_b, top1_b, "BM25 (상대 분리도)", "#3D484A")]:
    order = s_.sort_values(ascending=False).index
    acc = t_[order].cumsum() / np.arange(1, len(order) + 1)
    axes[1].plot(np.arange(1, len(order) + 1) / len(order), acc.values, color=c, label=name)
    axes[1].axhline(t_.mean(), color=c, ls=":", alpha=0.7)
axes[1].set_xlabel("분리도 높은 순으로 답한 질문 비율 (커버리지)"); axes[1].set_ylabel("그중 1위가 정답인 비율")
axes[1].legend(); axes[1].set_title("분리도가 큰 질문만 답하면 정확도가 오른다")
plt.tight_layout(); plt.show()

**관찰:** 분리도 상위 20% 질문만 답하면 벡터 검색의 1위 정확도가 약 0.7 로, 전체 평균(점선 0.25)의 세 배 가까이 된다.
BM25 도 같은 경향이다(상위 40% 에서 거의 1.0).
**결과의 모양 자체가 품질 신호**다. 분리도가 낮으면 "모르겠다"고 하거나, 다른 검색(BM25)으로 넘기거나, 질의를 다시 쓰게 할 수 있다.

## 4. 유사도의 절대값은 믿을 수 없다 — 이방성
아무 관련 없는 질문–청크 쌍의 코사인은 0 근처일까?

In [ ]:
qe = np.stack(q(conn, "SELECT embedding FROM questions ORDER BY random() LIMIT 500")["embedding"].to_numpy())
ce = np.stack(q(conn, "SELECT embedding FROM doc_chunks ORDER BY random() LIMIT 3000")["embedding"].to_numpy())
qe /= np.linalg.norm(qe, axis=1, keepdims=True); ce /= np.linalg.norm(ce, axis=1, keepdims=True)
rand_pairs = (qe @ ce.T).ravel()
gold_sims = [sims[qid][rank[qid] - 1] for qid in QIDS if rank[qid]]

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.hist(rand_pairs, bins=80, density=True, alpha=0.6, color="#9aa5a8", label=f"무작위 질문–청크 쌍 (평균 {rand_pairs.mean():.2f})")
ax.hist(gold_sims, bins=40, density=True, alpha=0.6, color="#FF7043", label=f"질문–정답 지문 (평균 {np.mean(gold_sims):.2f})")
ax.axvline(0, color="black", lw=0.8); ax.legend(); ax.set_xlabel("코사인 유사도")
ax.set_title("관련 없는 쌍도 0 이 아니다 — 임베딩이 한쪽으로 쏠려 있다(이방성)")
plt.show()

관련 없는 쌍의 유사도가 0 이 아니라 평균 0.15 의 양수 쪽에 넓게 퍼져 있고, 0.4 를 넘는 쌍도 있어 정답 쌍 분포와 겹친다.
이 위치는 임베딩 모델마다 다르다(multilingual-e5 계열은 관련 없는 쌍도 0.7 안팎으로 높게 나오는 것으로 알려져 있다).
그래서 "유사도 0.6 이상만" 같은 **고정 임계값은 모델마다 의미가 다르다.** 절대값보다 곡선 모양(2·3절)을 본다.

## 5. 필터 적합성 — 필터로 좁힌 집합이 질의와 맞는가
뉴스 지문에는 분야(category)가 있다. 정답 지문이 뉴스인 질문에 대해 세 가지로 검색한다.
- 필터 없음 · 정답 분야로 필터(맞는 필터) · 엉뚱한 분야로 필터(틀린 필터)

In [ ]:
cat = q(conn, "SELECT id, category FROM docs WHERE category IS NOT NULL").set_index("id")["category"]
cat_size = cat.value_counts()
news_q = [qid for qid in QIDS if runs["gold"][qid] in cat.index][:200]
qemb = q(conn, "SELECT id, embedding FROM questions WHERE id = ANY(%s)", (news_q,)).set_index("id")["embedding"]
rng = np.random.default_rng(0)

SQL = """WITH c AS MATERIALIZED (
           SELECT ch.doc_id, ch.embedding FROM doc_chunks ch JOIN docs d ON d.id = ch.doc_id WHERE d.category = %(c)s)
         SELECT doc_id, max(1 - (embedding <=> %(e)s)) AS sim FROM c GROUP BY doc_id ORDER BY sim DESC LIMIT 10"""
rows = []
for qid in news_q:
    g = runs["gold"][qid]; right = cat[g]
    wrong = rng.choice([c for c in cat_size.index[:15] if c != right])
    base = runs["vec_chunk"][qid][:10]
    rows.append({"필터": "필터 없음", "top1 유사도": base[0][1], "hit": g in [d for d, _ in base], "후보 지문 수": len(cat) + 1625})
    for name, cc in [("맞는 분야", right), ("틀린 분야", wrong)]:
        r = q(conn, SQL, {"c": cc, "e": qemb[qid]})
        rows.append({"필터": name, "top1 유사도": r["sim"].iloc[0], "hit": g in r["doc_id"].tolist(), "후보 지문 수": cat_size[cc]})
fr = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
colors = {"필터 없음": "#9aa5a8", "맞는 분야": "#009CA6", "틀린 분야": "#FF7043"}
for name, g_ in fr.groupby("필터", sort=False):
    axes[0].hist(g_["top1 유사도"], bins=30, alpha=0.55, color=colors[name], label=name)
axes[0].legend(); axes[0].set_xlabel("1위 유사도"); axes[0].set_title("틀린 필터 → 1위 유사도가 확 떨어진다")
hit = fr.groupby("필터", sort=False)["hit"].mean()
axes[1].bar(hit.index, hit.values, color=[colors[k] for k in hit.index])
for i, v in enumerate(hit.values):
    axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom")
axes[1].set_ylim(0, 1.05); axes[1].set_title("Hit@10")
axes[2].barh(cat_size.index[:15][::-1], cat_size.values[:15][::-1], color="#3D484A")
axes[2].set_title("분야별 지문 수 (상위 15) — 필터 후 남는 후보")
plt.tight_layout(); plt.show()

**관찰:**
- 맞는 필터는 후보를 수백 개로 줄여 Hit@10 을 0.45 → 0.71 로 올린다 (틀린 필터는 정답이 후보에 아예 없으니 0)
- 틀린 필터는 1위 유사도부터 뚝 떨어진다 — 슬라이드 "필터 집합의 유사도가 낮다면 시맨틱 불일치". **1위 유사도 비교만으로 필터가 틀렸음을 감지할 수 있다**
- 분야가 작을수록 후보가 적어 벡터 검색은 거의 "목록 전체 보기"가 된다 (슬라이드 "너무 적으면 시맨틱 검색 무력화")

## 6. 코퍼스 분포 — 결과가 특정 문서로 쏠리는가 (허브)

In [ ]:
def appear(method, k=10):
    cnt = {}
    for qid in QIDS:
        for d, _ in runs[method][qid][:k]:
            cnt[d] = cnt.get(d, 0) + 1
    return pd.Series(cnt).sort_values(ascending=False)


app = {"BM25": appear("bm25"), "벡터 (청크)": appear("vec_chunk")}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
for (name, s), c in zip(app.items(), ["#3D484A", "#FF7043"]):
    axes[0].plot(np.arange(1, len(s) + 1), s.values, color=c, label=f"{name}: 서로 다른 지문 {len(s):,}개")
axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].legend()
axes[0].set_xlabel("지문 (등장 횟수 순)"); axes[0].set_ylabel("500개 질문의 top-10 에 등장한 횟수")
axes[0].set_title("벡터 검색에는 '어디에나 나오는' 허브 지문이 있다")

src = q(conn, "SELECT id, source FROM docs").set_index("id")["source"]
share = pd.DataFrame({
    "정답 지문": pd.Series([src[runs["gold"][qid]] for qid in QIDS]).value_counts(normalize=True),
    "BM25 1위": pd.Series([src[runs["bm25"][qid][0][0]] for qid in QIDS]).value_counts(normalize=True),
    "벡터 1위": pd.Series([src[runs["vec_chunk"][qid][0][0]] for qid in QIDS]).value_counts(normalize=True),
}).fillna(0).T
share.plot.barh(stacked=True, ax=axes[1], color=["#009CA6", "#FF7043", "#3D484A"])
axes[1].set_xlabel("비율"); axes[1].set_title("출처 분포: 정답 vs 1위 결과"); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

hubs = app["벡터 (청크)"].head(8)
titles = q(conn, "SELECT id, title, source FROM docs WHERE id = ANY(%s)", (hubs.index.tolist(),)).set_index("id")
pd.DataFrame({"등장 횟수": hubs, "제목": titles.loc[hubs.index, "title"], "출처": titles.loc[hubs.index, "source"]})

**관찰:**
- 벡터 검색에서는 몇몇 지문이 수십 개 질문의 top-10 에 반복해 나온다(허브). 고차원 임베딩의 알려진 성질이다
- 허브 지문은 질문과 상관없이 "평균적으로 비슷해" 보이는 지문이다 → 결과 다양성을 떨어뜨리고 진짜 정답을 밀어낸다
- 출처 분포가 정답 분포와 다르면 코퍼스나 청킹이 한쪽에 유리하게 되어 있다는 신호다 (슬라이드 "코퍼스 커버리지")

**정리:** 30번이 "얼마나 맞히나"라면, 31번은 **"맞힐 수 있는 모양인가"** 를 본다. 정답셋 없이도 운영 중에 잴 수 있는 신호들이다.